# Tutorial 09 — Protein interface analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uw-ipd/tmol/blob/master/docs/tutorial/09_protein_interface_hotspot_scan.ipynb)

Which residues make favorable contacts across a protein interface? Map their contributions, choose three sites, and replace each with alanine while letting its neighbors repack.

We use chains A and B of the KcsA potassium channel (1BL8). This examines one interface, without the rest of the tetramer or its membrane environment. Start with [scoring](03_scoring_and_analysis.ipynb) and [packing](04_packing_and_mutation_scan.ipynb).


## Setup

In Colab, choose a **GPU runtime** and **Run all**. For local execution, follow the [installation guide](../installation.md) with the documentation extras. Run cells in order; setup selects CUDA when available, otherwise CPU.

The example uses chains A and B from 1BL8, a neighboring pair in the potassium-channel tetramer.


In [ ]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlopen

    exec(
        urlopen(
            "https://raw.githubusercontent.com/uw-ipd/tmol/"
            "master/docs/tutorial/colab_setup.py"
        ).read(),
        globals(),
    )
    setup_colab(["tmol/tests/data/cif/1BL8.cif"])


In [ ]:
import logging

# The example files do not need local PDB or CCD mirrors.
logging.getLogger("atomworks.constants").addFilter(
    lambda record: not record.getMessage().startswith(
        (
            "Environment variable CCD_MIRROR_PATH not set.",
            "Environment variable PDB_MIRROR_PATH not set.",
        )
    )
)

from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path
from time import perf_counter
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from biotite.structure.io import load_structure

import tmol
from tmol.database import ParameterDatabase
from tmol.io import pose_stack_from_biotite
from tmol.ops import (
    calculate_block_pair_ddg,
    compute_block_adjacency,
    compute_block_centroids_and_furthest_dist,
    res_mask_to_coord_mask,
)
from tmol.pack import PackerPalette, PackerTask, pack_rotamers
from tmol.pack.rotamer import FixedAAChiSampler, IncludeCurrentSampler
from tmol.pack.rotamer.dunbrack import create_dunbrack_sampler_from_database
from tmol.pose import PoseStackBuilder
from tmol.score import ScoreType, beta2016_score_function

SEED = 20260829
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings(
    "ignore", message=r"Sparse invariant checks are implicitly disabled.*"
)

device = (
    torch.device("cuda", torch.cuda.current_device())
    if tmol.has_cuda() and torch.cuda.is_available()
    else torch.device("cpu")
)
from itables import show as show_table

# Format displayed numbers without changing the values used in calculations.
pd.options.display.float_format = "{:.4g}".format


def block_label(pose, block_index):
    chain = pose.pdb_info.chain_labels[0, block_index]
    residue = pose.pdb_info.residue_labels[0, block_index]
    insertion = pose.pdb_info.residue_insertion_codes[0, block_index]
    return f"{chain}:{residue}{insertion}"


# Locate the example files.
repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "tmol/tests/data").is_dir()
    ),
    Path(tmol.__file__).resolve().parents[1],
)
if not (repo_root / "tmol/tests/data/cif/1BL8.cif").exists():
    repo_root = Path(tmol.__file__).resolve().parents[1]
cif_path = repo_root / "tmol/tests/data/cif/1BL8.cif"


## Load chains A and B

Keep protein atoms from the two neighboring subunits and optimize their hydrogens. Use the author chain labels to build the masks for subsequent scoring.


In [ ]:
atom_array = load_structure(str(cif_path), model=1, include_bonds=True)
protein_dimer = atom_array[
    ((atom_array.chain_id == "A") | (atom_array.chain_id == "B"))
    & ~atom_array.hetero
]
param_db = ParameterDatabase.get_default()
with redirect_stdout(StringIO()), redirect_stderr(StringIO()):
    pose_stack = pose_stack_from_biotite(
        protein_dimer,
        device,
        param_db=param_db,
        no_optH=False,
    )

score_function = beta2016_score_function(device, param_db=param_db)


chain_labels = pose_stack.pdb_info.chain_labels[0]
chain_a = torch.tensor(
    [str(label) == "A" for label in chain_labels], device=device
)
chain_b = torch.tensor(
    [str(label) == "B" for label in chain_labels], device=device
)
assert int(chain_a.sum()) == int(chain_b.sum()) == 97
print(
    f"device={device}; input={cif_path.name}; "
    f"blocks={pose_stack.max_n_blocks}; chains=A+B"
)


## Inspect the interface

Chain A is blue; chain B is its neighboring subunit. The chain labels from the input define the partners throughout this notebook.


In [ ]:
chain_a_atoms = res_mask_to_coord_mask(pose_stack, chain_a.unsqueeze(0))
interface_viewer = tmol.view(
    pose_stack,
    highlighted=chain_a_atoms[pose_stack.real_atoms],
    highlight_color="#2563eb",
)
print("Blue: chain A; unhighlighted partner: chain B.")
interface_viewer.show()


## Map interactions across the two chains

Compute the block-pair score matrix, then add A→B and B→A entries. Rank residues using five nonbonded terms: attraction, repulsion, solvation, electrostatics, and hydrogen bonding.

In the heat map, negative values are favorable under this weighted sum. These are contributions in one structure, not experimental residue effects. A separate all-term sum is checked against `calculate_block_pair_ddg()`.


In [ ]:
block_scorer = score_function.render_block_pair_scoring_module(pose_stack)
weighted_by_term = block_scorer(
    pose_stack.coords, sum_terms=False, apply_weights=True
)
score_types = score_function.all_score_types()

forward = weighted_by_term[:, 0][:, chain_a][:, :, chain_b]
reverse = weighted_by_term[:, 0][:, chain_b][:, :, chain_a].transpose(1, 2)
pair_by_term = forward + reverse
all_term_interface = pair_by_term.sum(dim=(1, 2))

chain_a_batch = chain_a.unsqueeze(0)
chain_b_batch = chain_b.unsqueeze(0)
helper_interface = calculate_block_pair_ddg(
    pose_stack,
    chain_a_batch,
    chain_b_batch,
    sfxn=score_function,
    sum_terms=False,
    minimize=False,
    pack=False,
)[:, 0]
torch.testing.assert_close(all_term_interface, helper_interface)


### Rank the nonbonded contributions

Sum the five selected terms across the partner chain for each residue.


In [ ]:
nonbonded_types = [
    ScoreType.fa_ljatr,
    ScoreType.fa_ljrep,
    ScoreType.fa_lk,
    ScoreType.fa_elec,
    ScoreType.hbond,
]
nonbonded_indices = [score_types.index(term) for term in nonbonded_types]
nonbonded_pairs = pair_by_term[nonbonded_indices].sum(dim=0)

chain_a_blocks = torch.nonzero(chain_a, as_tuple=False).flatten()
chain_b_blocks = torch.nonzero(chain_b, as_tuple=False).flatten()
chain_a_contributions = nonbonded_pairs.sum(dim=1)
chain_b_contributions = nonbonded_pairs.sum(dim=0)


Attach author residue labels to the contributions and plot the cross-chain matrix.


In [ ]:
def contribution_rows(blocks, contributions):
    rows = []
    for local_index, block_index_tensor in enumerate(blocks):
        block_index = int(block_index_tensor)
        rows.append(
            {
                "block_index": block_index,
                "author_label": block_label(pose_stack, block_index),
                "identity": pose_stack.block_type(0, block_index).name3,
                "five_term_interface_score": float(
                    contributions[local_index].detach().cpu()
                ),
            }
        )
    return rows


contribution_frame = pd.DataFrame(
    contribution_rows(chain_a_blocks, chain_a_contributions)
    + contribution_rows(chain_b_blocks, chain_b_contributions)
).sort_values("five_term_interface_score")
show_table(contribution_frame.head(16))

fig, ax = plt.subplots(figsize=(8.5, 6.5))
image = ax.imshow(
    nonbonded_pairs.detach().cpu(),
    cmap="coolwarm",
    aspect="auto",
    vmin=-max(1.0, float(nonbonded_pairs.abs().quantile(0.99).detach().cpu())),
    vmax=max(1.0, float(nonbonded_pairs.abs().quantile(0.99).detach().cpu())),
)
ax.set(
    xlabel="chain B block index within partner",
    ylabel="chain A block index within partner",
    title="KcsA A–B nonbonded block-pair accounting",
)
fig.colorbar(image, ax=ax, label="weighted score units")
plt.tight_layout()
plt.show()


## Choose three mutation sites

Select the three most favorable internal residues of chain A, excluding alanine and glycine. Their fixed-structure ranking tells us where to start; repacking may change that ranking.


In [ ]:
chain_a_block_list = chain_a_blocks.detach().cpu().tolist()
eligible = contribution_frame[
    (contribution_frame["author_label"].str.startswith("A:"))
    & ~contribution_frame["identity"].isin({"ALA", "GLY"})
    & contribution_frame["block_index"].isin(chain_a_block_list[1:-1])
]
hotspot_frame = eligible.head(3).copy().reset_index(drop=True)
hotspot_blocks = hotspot_frame["block_index"].astype(int).tolist()
assert len(hotspot_blocks) == 3
show_table(hotspot_frame)

hotspot_mask = torch.zeros_like(chain_a)
hotspot_mask[hotspot_blocks] = True
hotspot_atoms = res_mask_to_coord_mask(pose_stack, hotspot_mask.unsqueeze(0))
hotspot_viewer = tmol.view(
    pose_stack,
    highlighted=hotspot_atoms[pose_stack.real_atoms],
    highlight_color="#f97316",
)
print("Orange hotspots: " + ", ".join(hotspot_frame["author_label"].tolist()))
hotspot_viewer.show()


## Pack a wild-type control and alanine variant for each site

Build six batch members: two states for each of three sites. Each pair uses the same local packing shell. Only the mutant target may change identity; its neighbors can repack, and blocks outside the shell stay fixed.

The shell uses enclosing residue spheres plus a 5 Å gap. It is a conservative neighborhood, not an exact atom-distance contact set.


In [ ]:
requests = [
    (block_index, state)
    for block_index in hotspot_blocks
    for state in ("repacked WT", "ALA")
]
scan_batch = PoseStackBuilder.from_poses([pose_stack] * len(requests), device)
scan_task = PackerTask(scan_batch, PackerPalette())

mutation_mask = torch.zeros_like(scan_batch.block_type_ind, dtype=torch.bool)
for pose_index, (block_index, state) in enumerate(requests):
    if state == "ALA":
        mutation_mask[pose_index, block_index] = True

# Restrictions only remove choices. Preserve the original identity everywhere
# except mutant targets, then restrict those targets to alanine.
scan_task.restrict_to_repacking(~mutation_mask)
scan_task.restrict_absent_name3s({"ALA"}, mutation_mask)


### Give each WT/mutant pair the same shell

Use neighbors of its target and disable packing outside that mask.


In [ ]:
centroids, radii = compute_block_centroids_and_furthest_dist(pose_stack)
adjacency = compute_block_adjacency(centroids, radii)
packing_shells = torch.zeros_like(scan_batch.block_type_ind, dtype=torch.bool)
for pose_index, (block_index, _) in enumerate(requests):
    packing_shells[pose_index] = adjacency[0, block_index]
    packing_shells[pose_index, block_index] = True
scan_task.disable_packing_by_block_mask(~packing_shells)

scan_task.add_conformer_sampler(
    create_dunbrack_sampler_from_database(param_db, device)
)
scan_task.add_conformer_sampler(FixedAAChiSampler())
scan_task.add_conformer_sampler(IncludeCurrentSampler())


### Pack all six states

Check each target identity afterward and verify that every non-target identity stayed unchanged.


In [ ]:
torch.manual_seed(
    SEED
)  # Seeds either backend; CPU and CUDA outcomes can differ.
if device.type == "cuda":
    torch.cuda.synchronize()
scan_start = perf_counter()
packed_scan = pack_rotamers(scan_batch, score_function, scan_task)
if device.type == "cuda":
    torch.cuda.synchronize()
scan_seconds = perf_counter() - scan_start

for pose_index, (target_block, state) in enumerate(requests):
    expected = (
        "ALA"
        if state == "ALA"
        else pose_stack.block_type(0, target_block).name3
    )
    assert packed_scan.block_type(pose_index, target_block).name3 == expected
    for block_index in range(pose_stack.max_n_blocks):
        if block_index != target_block:
            assert (
                packed_scan.block_type(pose_index, block_index).name3
                == pose_stack.block_type(0, block_index).name3
            )

print(
    f"packed {len(requests)} site/state members in {scan_seconds:.3f} s; "
    f"shell sizes={packing_shells.sum(dim=1).detach().cpu().tolist()}"
)


## Compare each mutant with its matched control

Subtract the repacked wild-type score for the same site and shell. Report three views: the entire A–B interface, the target's interactions with B, and the whole-pose total.

For target-to-B scores, pass one target block index per batch member and a mask for chain B. This avoids building a dense target mask.


In [ ]:
chain_a_scan = chain_a.unsqueeze(0).expand(len(requests), -1).clone()
chain_b_scan = chain_b.unsqueeze(0).expand(len(requests), -1).clone()
interface_scores = calculate_block_pair_ddg(
    packed_scan,
    chain_a_scan,
    chain_b_scan,
    sfxn=score_function,
    minimize=False,
    pack=False,
)


### Isolate each target’s interactions with chain B

Request per-term values so the largest changes can be inspected later.


In [ ]:
target_indices = torch.tensor(
    [block_index for block_index, _ in requests],
    dtype=torch.int64,
    device=device,
)
target_partner_by_term = calculate_block_pair_ddg(
    packed_scan,
    target_indices,
    chain_b_scan,
    sfxn=score_function,
    sum_terms=False,
    minimize=False,
    pack=False,
)
target_partner_scores = target_partner_by_term.sum(dim=0)

whole_scorer = score_function.render_whole_pose_scoring_module(packed_scan)
whole_scores = whole_scorer(packed_scan.coords)


### Match mutant and control rows

Subtract only the WT state for that same site. Keep the structures and captions for visual comparison.


In [ ]:
comparison_rows = []
term_change_rows = []
variant_poses = {}
variant_notes = {}
for site_index, target_block in enumerate(hotspot_blocks):
    wt_index = 2 * site_index
    mutant_index = wt_index + 1
    author_label = block_label(pose_stack, target_block)
    native_identity = pose_stack.block_type(0, target_block).name3
    comparison_rows.append(
        {
            "site": author_label,
            "mutation": f"{native_identity}→ALA",
            "packing_shell_blocks": int(packing_shells[wt_index].sum()),
            "interface_score_change": float(
                (interface_scores[mutant_index] - interface_scores[wt_index])
                .detach()
                .cpu()
            ),
            "target_partner_score_change": float(
                (
                    target_partner_scores[mutant_index]
                    - target_partner_scores[wt_index]
                )
                .detach()
                .cpu()
            ),
            "whole_pose_score_change": float(
                (whole_scores[mutant_index] - whole_scores[wt_index])
                .detach()
                .cpu()
            ),
        }
    )
    for term_index, score_type in enumerate(score_types):
        term_change_rows.append(
            {
                "site": author_label,
                "term": score_type.name,
                "target_partner_score_change": float(
                    (
                        target_partner_by_term[term_index, mutant_index]
                        - target_partner_by_term[term_index, wt_index]
                    )
                    .detach()
                    .cpu()
                ),
            }
        )
    for member_index, state in (
        (wt_index, "repacked WT"),
        (mutant_index, "ALA"),
    ):
        key = f"{author_label} · {state}"
        variant_poses[key] = packed_scan.split(member_index)
        variant_notes[key] = (
            f"{native_identity}→{packed_scan.block_type(member_index, target_block).name3}; "
            f"same {int(packing_shells[member_index].sum())}-block packing shell"
        )

comparison_frame = pd.DataFrame(comparison_rows).sort_values(
    "interface_score_change", ascending=False
)
show_table(
    comparison_frame[
        [
            "site",
            "mutation",
            "packing_shell_blocks",
            "interface_score_change",
            "target_partner_score_change",
            "whole_pose_score_change",
        ]
    ].rename(
        columns={
            "site": "Site",
            "mutation": "Mutation",
            "packing_shell_blocks": "Shell size",
            "interface_score_change": "Interface Δ",
            "target_partner_score_change": "Target–B Δ",
            "whole_pose_score_change": "Total Δ",
        }
    )
)


Plot the three score changes relative to each site’s repacked WT. Positive values mean alanine is less favorable under that score; negative values mean it is more favorable.


In [ ]:
plot_frame = comparison_frame.assign(
    site_and_mutation=comparison_frame["site"]
    + " "
    + comparison_frame["mutation"]
).set_index("site_and_mutation")[
    [
        "interface_score_change",
        "target_partner_score_change",
        "whole_pose_score_change",
    ]
]
ax = plot_frame.plot.barh(figsize=(9, 4.5))
ax.axvline(0.0, color="black", linewidth=0.8)
ax.set(
    xlabel="alanine minus independently repacked WT (weighted score units)",
    ylabel="",
    title="Matched local-repacking comparisons",
)
ax.legend(
    ["chain A–B interface", "target–chain B", "whole pose"],
    loc="best",
)
plt.tight_layout()
plt.show()


The table identifies the five largest target-to-partner term changes at each site. Then compare that site’s WT and mutant structures in the viewer.


In [ ]:
term_change_frame = pd.DataFrame(term_change_rows)
term_change_frame["absolute_change"] = term_change_frame[
    "target_partner_score_change"
].abs()
dominant_term_changes = (
    term_change_frame.sort_values("absolute_change", ascending=False)
    .groupby("site", as_index=False, group_keys=False)
    .head(5)
    .sort_values(["site", "absolute_change"], ascending=[True, False])
)
show_table(dominant_term_changes.drop(columns="absolute_change"))

display(
    tmol.switchable_view(
        variant_poses,
        notes=variant_notes,
        width=760,
        height=440,
    )
)


Compare the WT and mutant structures for each site. Neighbors can rearrange, so a strong starting contact need not produce the largest mutation effect. One packed outcome per state does not establish a reliable ranking or binding ΔΔG.

## Try it

1. Add more sites and compare one batch with chunked batches.
2. Repeat each WT/mutant pair with several seeds and plot the score distributions.
3. Minimize matching local masks after packing and see whether the ranking changes.

## References

- Doyle et al., [The structure of the potassium channel](https://pubmed.ncbi.nlm.nih.gov/9525859/)
